# 🛒 Module 5: Market Basket Analysis (FP-Growth / Apriori)

In this notebook, we construct a Transaction Item Matrix from cleaned retail orders, execute the **FP-Growth Algorithm** to find frequent itemsets, generate **Association Rules**, and evaluate metrics (**Support, Confidence, Lift**).

---

In [ ]:
import os
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules
import plotly.express as px

cleaned_path = os.path.join("..", "data", "processed", "cleaned_retail_data.csv")
df = pd.read_csv(cleaned_path)
print(f"Loaded Cleaned Data: {df.shape[0]:,} rows")

### Step 1: Construct One-Hot Encoded Basket Matrix

In [ ]:
top_items = df['Description'].value_counts().head(1200).index
df_filtered = df[df['Description'].isin(top_items)]

basket = (df_filtered.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

basket_sets = (basket > 0)
print(f"Transaction Matrix Shape: {basket_sets.shape}")

### Step 2: FP-Growth Frequent Itemsets

In [ ]:
frequent_itemsets = fpgrowth(basket_sets, min_support=0.01, use_colnames=True)
print(f"Found {len(frequent_itemsets)} frequent itemsets.")
frequent_itemsets.head(10)

### Step 3: Generate Association Rules (Support, Confidence, Lift)

In [ ]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0.2)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
print(f"Generated {len(rules)} Association Rules!")

# Format frozenset to strings for clean viewing
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].head(20)

### Step 4: Export Rules to CSV

In [ ]:
rules_to_save = rules.copy()
rules_to_save['antecedents'] = rules_to_save['antecedents_str']
rules_to_save['consequents'] = rules_to_save['consequents_str']
rules_to_save.drop(columns=['antecedents_str', 'consequents_str'], inplace=True)

out_rules = os.path.join("..", "data", "processed", "association_rules.csv")
rules_to_save.to_csv(out_rules, index=False)
print(f"Association rules exported to: {out_rules}")